In [1]:
# Mount Google Drive to access the project files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports
import os
import ast

import numpy as np
import pandas as pd
import regex as re
from sklearn.model_selection import train_test_split


# File paths
FR_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/classified_fr_output.csv"
DE_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/classified_de_output.csv"
MIX_PATH = "/content/drive/MyDrive/code_switch_project/data/processed/mix_full_labeled.csv"

OUTPUT_DIR = "/content/drive/MyDrive/code_switch_project/data/final_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Dataset settings
TOP_N = 100_000       # Number of French and German examples to keep
RANDOM_STATE = 42     # Ensures reproducible sampling and splits

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

In [3]:
# Cleaning function for the French-German mixed dataset

LETTER = re.compile(r"\p{L}")


def clean_sentence(text):
    """
    Clean SwitchLingua examples while preserving sentence content.
    """

    if pd.isna(text):
        return None

    text = str(text).strip()

    # Convert list-like strings into normal text
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            text = " ".join(str(x) for x in parsed)
    except:
        pass

    # Remove square brackets but keep their content
    text = text.replace("[", "").replace("]", "")

    # Remove parenthesized metadata or noise
    text = re.sub(r"\([^)]*\)", "", text)

    # Normalize whitespace
    text = " ".join(text.split())

    # Remove empty rows and rows without letters
    if not text:
        return None

    if not LETTER.search(text):
        return None

    return text

## Select Chat Examples and Prepare the Mixed Dataset

Load the classified French and German datasets and retain the highest-confidence chat examples. The French-German mixed dataset is loaded separately and cleaned using the SwitchLingua-specific cleaning function.

In [4]:
# Load and filter French chat examples
print("Loading French dataset...")
fr = pd.read_csv(FR_PATH, usecols=['text', 'predicted_label', 'confidence'])

fr_chat = fr[fr['predicted_label'] == 'chat'].copy()
fr_chat = fr_chat.nlargest(TOP_N, 'confidence')
fr_chat['language'] = 'french'

print(f"FR chat kept: {len(fr_chat):,} rows (top {TOP_N} by confidence)")

del fr  # Free memory


# Load and filter German chat examples
print("\nLoading German dataset...")
de = pd.read_csv(DE_PATH, usecols=['text', 'predicted_label', 'confidence'])

de_chat = de[de['predicted_label'] == 'chat'].copy()
de_chat = de_chat.nlargest(TOP_N, 'confidence')
de_chat['language'] = 'german'

print(f"DE chat kept: {len(de_chat):,} rows (top {TOP_N} by confidence)")

del de  # Free memory


# Load and clean French-German mixed examples
print("\nLoading Mixed dataset...")
mix = pd.read_csv(MIX_PATH)

mix = mix.rename(columns={
    'label': 'predicted_label',
    'score': 'confidence'
})

mix['language'] = 'mixed'
mix['text'] = mix['text'].apply(clean_sentence)
mix = mix.dropna(subset=['text'])

print(f"Mix rows: {len(mix):,}")

Loading French dataset...
FR chat kept: 100,000 rows (top 100000 by confidence)

Loading German dataset...
DE chat kept: 100,000 rows (top 100000 by confidence)

Loading Mixed dataset...
Mix rows: 8,511


## Synthetic Mixed Data Generation

To increase the amount of French-German code-switched data, synthetic mixed examples were created from the high-confidence French and German chat datasets.

Common French-German word and phrase equivalents were defined and randomly substituted into the sentences. Most sentences received one substitution, while a smaller proportion received two or three substitutions.

In [5]:
# French-to-German word and phrase replacements
FR_TO_DE_SWAPS = {
    # Transport
    'gare': 'Hbf',
    'train': 'Zug',
    'ticket': 'Ticket',
    'bus': 'Bus',
    'métro': 'U-Bahn',
    'tram': 'Straßenbahn',
    'station': 'Bahnhof',
    'ligne': 'Linie',
    'voiture': 'Auto',
    'vélo': 'Fahrrad',
    'arrêt': 'Haltestelle',

    # University
    'université': 'Uni',
    'faculté': 'Fakultät',
    'cours': 'Vorlesung',
    'examen': 'Klausur',
    'devoir': 'Hausarbeit',
    'étudiant': 'Student',
    'études': 'Studium',
    'semestre': 'Semester',
    'td': 'Übung',
    'tp': 'Praktikum',
    'bibliothèque': 'Bibliothek',
    'amphi': 'Hörsaal',
    'licence': 'Bachelor',

    # Administration / bureaucracy
    'rendez-vous': 'Termin',
    'mairie': 'Bürgeramt',
    'préfecture': 'Ausländerbehörde',
    'titre de séjour': 'Aufenthaltstitel',
    'visa': 'Visum',
    'formulaire': 'Formular',
    'demande': 'Antrag',
    'inscription': 'Anmeldung',
    'désinscription': 'Abmeldung',
    'document': 'Dokument',
    'passeport': 'Reisepass',
    "carte d'identité": 'Personalausweis',
    'adresse': 'Adresse',
    'signature': 'Unterschrift',

    # Housing
    'appartement': 'Wohnung',
    'studio': 'WG',
    'bail': 'Mietvertrag',
    'loyer': 'Miete',
    'propriétaire': 'Vermieter',
    'caution': 'Kaution',
    'charges': 'Nebenkosten',
    'quartier': 'Viertel',
    'voisin': 'Nachbar',

    # Healthcare
    'docteur': 'Hausarzt',
    'hôpital': 'Krankenhaus',
    'pharmacie': 'Apotheke',
    'assurance': 'Krankenkasse',
    'urgence': 'Notaufnahme',
    'ordonnance': 'Rezept',
    'rendez-vous médical': 'Arzttermin',

    # Work
    'travail': 'Job',
    'stage': 'Praktikum',
    'contrat': 'Arbeitsvertrag',
    'job étudiant': 'Werkstudent',
    'salaire': 'Gehalt',
    'patron': 'Chef',
    'collègue': 'Kollege',
    'bureau': 'Büro',
    'réunion': 'Meeting',
    'candidature': 'Bewerbung',

    # Daily chat
    'maintenant': 'jetzt',
    'demain': 'morgen',
    'hier': 'gestern',
    "aujourd'hui": 'heute',
    'toujours': 'immer',
    'jamais': 'nie',
    'encore': 'noch',
    'déjà': 'schon',
    'bientôt': 'bald',
    'tard': 'spät',
    'tôt': 'früh',
    'vite': 'schnell',
    'bien': 'gut',
    'mal': 'schlecht',
    'beaucoup': 'viel',
    'peut-être': 'vielleicht',
    'ensemble': 'zusammen',
    'seul': 'allein',
    'oui': 'ja',
    'non': 'nein',
    'merci': 'danke',
    'bonjour': 'hallo',
    'salut': 'hey',
    'bonsoir': 'guten abend',
    'au revoir': 'tschüss',
    'désolé': 'entschuldigung',
    'excusez-moi': 'entschuldigung',
    "s'il te plaît": 'bitte',
    "d'accord": 'okay',
    'super': 'super',
    'génial': 'toll',
    'sympa': 'nett',
    'dommage': 'schade',

    # Food
    'supermarché': 'Supermarkt',
    'boulangerie': 'Bäckerei',
    'repas': 'Essen',
    'déjeuner': 'Mittagessen',
    'dîner': 'Abendessen',
    'petit-déjeuner': 'Frühstück',

    # Technology / chat
    'email': 'Mail',
    'message': 'Nachricht',
    'téléphone': 'Handy',
    'application': 'App',
    'réseau': 'Netzwerk',
    'mot de passe': 'Passwort',

    # Police / security
    'police': 'Polizei',
    'commissariat': 'Polizeirevier',
    'agent': 'Beamter',
    'plainte': 'Anzeige',
    'garde à vue': 'Gewahrsam',
    'interpellation': 'Festnahme',
    'témoin': 'Zeuge',
    'suspect': 'Verdächtiger',
    'enquête': 'Ermittlung',
    'rapport': 'Bericht',
}

# German-to-French replacements
DE_TO_FR_SWAPS = {v: k for k, v in FR_TO_DE_SWAPS.items()}

In [6]:
def choose_n_swaps():
    """
    Randomly choose how many words or phrases to replace.
    """

    return np.random.choice(
        [1, 2, 3],
        p=[0.80, 0.15, 0.05]
    )


def create_synthetic_mixed(text, swap_dict, n_swaps=1):
    """
    Replace selected words or phrases with equivalents from the other language.
    Returns None if no replacement is possible.
    """

    # Skip very short sentences
    if len(text.split()) < 3:
        return None

    words = swap_dict.keys()

    found = [
        word for word in words
        if re.search(r'\b' + re.escape(word) + r'\b', text, re.IGNORECASE)
    ]

    if not found:
        return None

    # Randomly select words or phrases to replace
    to_swap = np.random.choice(
        found,
        size=min(n_swaps, len(found)),
        replace=False
    )

    result = text

    for word in to_swap:
        replacement = swap_dict[word]
        result = re.sub(
            r'\b' + re.escape(word) + r'\b',
            replacement,
            result,
            flags=re.IGNORECASE
        )

    return result

In [7]:
# Generate synthetic mixed examples from French chat data
print("Generating synthetic mixed from French...")

np.random.seed(RANDOM_STATE)

fr_sample = fr_chat.sample(
    n=min(100000, len(fr_chat)),
    random_state=RANDOM_STATE
)

synthetic_from_fr = []

for text in fr_sample['text']:
    n_swaps = choose_n_swaps()

    result = create_synthetic_mixed(
        str(text),
        FR_TO_DE_SWAPS,
        n_swaps=n_swaps
    )

    if result is not None:
        synthetic_from_fr.append(result)

print(f"Synthetic mixed from French: {len(synthetic_from_fr):,}")

Generating synthetic mixed from French...
Synthetic mixed from French: 19,749


In [8]:
# Generate synthetic mixed examples from German chat data
print("Generating synthetic mixed from German...")

de_sample = de_chat.sample(
    n=min(100000, len(de_chat)),
    random_state=RANDOM_STATE
)

synthetic_from_de = []

for text in de_sample['text']:
    n_swaps = choose_n_swaps()

    result = create_synthetic_mixed(
        str(text),
        DE_TO_FR_SWAPS,
        n_swaps=n_swaps
    )

    if result is not None:
        synthetic_from_de.append(result)

print(f"Synthetic mixed from German: {len(synthetic_from_de):,}")

Generating synthetic mixed from German...
Synthetic mixed from German: 36,672


In [9]:
# Combine synthetic examples into one mixed dataset
synthetic_df = pd.DataFrame({
    'text': synthetic_from_fr + synthetic_from_de,
    'predicted_label': 'chat',
    'confidence': 1.0,
    'language': 'mixed'
})

print(f"\nTotal synthetic mixed examples: {len(synthetic_df):,}")
print(f"Original mixed examples: {len(mix):,}")
print(f"Total mixed after combining: {len(mix) + len(synthetic_df):,}")


Total synthetic mixed examples: 56,421
Original mixed examples: 8,511
Total mixed after combining: 64,932


In [10]:
# Check the first few rows of each dataset
print(fr_chat.head())
print(de_chat.head())
print(mix.head())
print(synthetic_df.head())

# Check the columns of each dataset
print(fr_chat.columns)
print(de_chat.columns)
print(mix.columns)
print(synthetic_df.columns)

                                                       text predicted_label  \
3992646   Si tu as confiance en toi, ta "Jambe Crocheteu...            chat   
38377739                            Axl, échange avec moi !            chat   
24656555  Ecoute, Teacup, vous devez me faire confiance,...            chat   
1329829       Il te pétera la gueule, tu fondras vite fait!            chat   
1330588      Il te pétera la gueule, tu fondras vite fait !            chat   

          confidence language  
3992646     0.999636   french  
38377739    0.999634   french  
24656555    0.999634   french  
1329829     0.999632   french  
1330588     0.999632   french  
                          text predicted_label  confidence language
15032649         Juhu, Lizzie?            chat    0.998784   german
1519229         Hallo, Schatz?            chat    0.998772   german
19110389      - Hallo, Schatz?            chat    0.998767   german
2693733          Cebe, Schatz?            chat    0.998765   

In [11]:
# ============================================================
# COMBINE FINAL DATASET
# ============================================================

# Keep only the text and language columns
fr_final = fr_chat[['text', 'language']]
de_final = de_chat[['text', 'language']]
mix_final = mix[['text', 'language']]
synthetic_final = synthetic_df[['text', 'language']]

# Combine French, German, original mixed, and synthetic mixed data
final_df = pd.concat(
    [
        fr_final,
        de_final,
        mix_final,
        synthetic_final
    ],
    ignore_index=True
)

# Shuffle the combined dataset
final_df = final_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print(f"Total dataset size: {len(final_df):,}")

# Check the distribution of language classes
print("\nClass distribution:")
print(final_df['language'].value_counts())

# Check the first few examples
print("\nFirst examples:")
print(final_df.head())

Total dataset size: 264,932

Class distribution:
language
german    100000
french    100000
mixed      64932
Name: count, dtype: int64

First examples:
                                                text language
0                Hey, willst du die Zufuhr wechseln?   german
1                                     - salut, Lucy.    mixed
2                               Hey, du gehst schon?   german
3                                - Écoute-moi, Léo !   french
4  - Votre ami Mikey a dit que vous lui parlez vi...    mixed


In [12]:
# Check the percentage distribution of each language class
final_df['language'].value_counts(normalize=True)

,proportion
language,
german,0.377455
french,0.377455
mixed,0.245089


In [13]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Split the dataset into 70% training, 15% validation, and 15% test data.
# Stratification preserves the language-class distribution in each split.

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    stratify=final_df['language'],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['language'],
    random_state=RANDOM_STATE
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df['language'].value_counts(normalize=True))

print("\nValidation distribution:")
print(val_df['language'].value_counts(normalize=True))

print("\nTest distribution:")
print(test_df['language'].value_counts(normalize=True))

Train: 185452
Validation: 39740
Test: 39740

Train distribution:
language
french    0.377456
german    0.377456
mixed     0.245088
Name: proportion, dtype: float64

Validation distribution:
language
german    0.377453
french    0.377453
mixed     0.245093
Name: proportion, dtype: float64

Test distribution:
language
french    0.377453
german    0.377453
mixed     0.245093
Name: proportion, dtype: float64


In [14]:
# ============================================================
# SAVE SPLITS
# ============================================================

# Save the training, validation, and test sets as separate CSV files.

train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test.csv"),
    index=False
)

print("\nDatasets saved successfully.")


Datasets saved successfully.
